# Footprint sources: local test

This notebook only imports from `core` and `app` and calls them. No geometry logic lives here.

Run it from the repository root, or keep the `sys.path` cell below.

In [ ]:
import sys
from pathlib import Path

REPO_ROOT = Path.cwd().parents[1]
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from city_information_modeler.app import footprint_collection
from city_information_modeler.core.sources import BdtreFootprintSource, DbgtFootprintSource
from city_information_modeler.parallel import Runner, load_profile

## The boundary to test on

Any GeoJSON in EPSG:4326: a bare Polygon, a Feature, or a FeatureCollection.

In [ ]:
boundary = {
    "type": "Polygon",
    "coordinates": [[
        [7.660, 45.055],
        [7.685, 45.055],
        [7.685, 45.070],
        [7.660, 45.070],
        [7.660, 45.055],
    ]],
}

## What this machine will do

`explain` shows how the runner would divide the call, without running it. The worker counts
come from `config/compute.yaml`, never from the core class.

In [ ]:
print(load_profile().describe())

source = DbgtFootprintSource()
runner = Runner()
runner.explain(source.fetch_buildings, boundary)

## Calling a core method directly

A direct call runs the whole job in this process. This is the plain way to test core code.

In [ ]:
buildings = source.fetch_buildings(boundary)
print(len(buildings), "buildings")
buildings.head()

## The same method through the runner

Same answer, split across workers. Switching to the `sequential` profile reproduces a
parallel run in one worker, which is the first thing to try when a result looks wrong.

In [ ]:
parallel_buildings = runner.run(source.fetch_buildings, boundary)
print(runner.last_report.describe())

print("same buildings as the direct call:",
      set(parallel_buildings["building_uuid"]) == set(buildings["building_uuid"]))

## Comparing the sources

`collect_all` is one orchestration in `app/`. Every source returns the same columns, so the
rows stack and the coverage of each attribute can be compared directly.

In [ ]:
everything = footprint_collection.collect_all(boundary)
footprint_collection.coverage(everything)

In [ ]:
import matplotlib.pyplot as plt

figure, axes = plt.subplots(1, 2, figsize=(16, 8))
for axis, source_name in zip(axes, ["bdtre", "dbgt"]):
    subset = everything[everything["source"] == source_name]
    subset.plot(ax=axis, column="usage", legend=False, alpha=0.8)
    axis.set_title(f"{source_name}  ({len(subset)} buildings)")
plt.tight_layout()
plt.show()

## Conflating the sources

One building per reference footprint. The first source in the priority defines the buildings
and supplies the geometry; the rest fill in what it left empty.

Read the match report first. If the sources disagree about where buildings are, no field
resolution can repair that, and the numbers below would be meaningless.

In [ ]:
from city_information_modeler.core.conflation import BuildingConflator, SourceFrames

PRIORITY = ("dbgt", "bdtre")

frames = footprint_collection.collect_by_source(boundary, PRIORITY)
conflator = BuildingConflator()
conflator.match_report(SourceFrames(frames=frames, priority=PRIORITY))

In [ ]:
conflated = footprint_collection.conflate_frames(frames, PRIORITY)
print(runner.last_report.describe())

footprint_collection.conflation_gain(conflated)

Every field group carries an `_origin` column naming the source it came from, so any value
in the result can be traced back.

In [ ]:
conflated[[
    "usage", "usage_origin",
    "height_m", "height_m_origin",
    "construction_year", "construction_year_origin",
    "matched_sources",
]].head(10)